## 03 · Input pixel noise only

Train a fresh four-loss model on the baseline COCO positives, changing only the image input with Gaussian pixel noise.

Notebook outline only: markdown cells describe the planned Colab implementation. No training code, data manifests, measurements, checkpoints, or scores are included. The experiment has not been run.

Reference: [MACCO source at `97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae`](https://github.com/hiker-lw/MACCO/tree/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae). The author code is under the [MACCO MIT license](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/LICENSE). When implementing, preserve original notices and keep the needed code visible in this notebook.


## S01 · Environment and assets

Record the GPU, CUDA availability, PyTorch and torchvision versions, and required packages (`ftfy`, `regex`, and the dependencies used by the selected author code). Stage chosen images under `/content/macco_data/`. Obtain the CLIP tokenizer vocabulary and original OpenAI ViT-B/32 weights as data assets. Keep implementation inside this notebook when code is added.


## S02 · Tokenizer

Use the pinned MACCO `src/open_clip_code/tokenizer.py` as the reference for the CLIP BPE helpers, `SimpleTokenizer`, and `tokenize`. Record an explicit path to `bpe_simple_vocab_16e6.txt.gz` and check positive and reviewed-negative captions use the same tokenizer.


## S03 · CLIP and MACCO model

Use ViT-B/32 with QuickGELU, the pinned CLIP and MACCO model definitions, and both image and text encoders. For every independent run, reload the same original OpenAI CLIP checkpoint, then initialize new MACCO predictors with the recorded common seed. Record the checkpoint identifier and hash. Train both encoders from the first optimizer step; use float32 trainable parameters with AMP.


## S04 · Preprocessing and datasets

Preserve the author's image transforms, crop-aware box-to-patch conversion, text masking, annotation fields, and joint image–text path. Select one positive row per source image or frame. Freeze row IDs before training. Shared COCO counts: 1,000 train from `train2014`, 500 project validation and 500 project test from disjoint `val2014` rows. Shared Talk2Car counts: 1,000 train, 500 official labeled validation, and 500 project test from whole scenes reserved from the official labeled training set. Check image, frame, and scene overlap.


## S05 · Original losses

Use the pinned author definitions, masking, normalization, targets, and reductions for MCA, MIR, MLM, and MIM. Baseline coefficients: MCA 1.0, MIR 0.1, MLM 20.0, MIM 250.0. Compare the four unweighted loss values on one fixed batch with the pinned author implementation during development.


## S06 · Training and checkpoints

Use the pinned single-GPU joint training path with a fresh optimizer per run, AMP, one update per batch, and `drop_last=False`. Train five full epochs over 1,000 positive pairs (5,000 pair presentations). Learning rates: `5e-7` for pretrained encoders and `1e-3` for new predictors; warmup: 50 updates. Choose one common batch size after the separate 8/16/32 speed check; candidate 16 gives 63 updates per epoch and 315 total. Compute the final update budget from the loader length. Save resolved settings, per-term losses, rows per epoch, update count, and checkpoint path.


## S07 · Shared evaluation

Freeze cases, candidate captions, preprocessing, scoring rules, and noise seed before final testing. Use the same 500-row Talk2Car validation and test pools for all five runs; COCO-trained runs also use the same COCO validation and test pools and separate selected Table 1 and Table 21 evaluations. Report image-to-text and text-to-image Recall@1, Recall@5, and Recall@10 over fixed candidate pools. Score clean and fixed-noise inputs on the same cases. Record benchmark, split, condition, seed, metric, unit, candidate pool, and `n_examples`. Compare variants with baseline only after matching these keys.


## 03.01 · Setup and shared inputs

Use this notebook's S01–S07 implementation and the exact baseline 1,000 COCO training rows. Keep original four loss weights and masks. Record a training noise strength and a separate training noise seed.


## 03.02 · Pixel transform

After crop and resize, add Gaussian noise to pixels in `[0, 1]`, clamp to that range, and then apply CLIP normalization. If the batch is already normalized, undo normalization before adding noise and reapply it afterward. A pilot sigma is 0.07. Check sigma zero, positive sigma, unchanged shape, and pre-normalization range.


## 03.03 · Training connection

Feed noisy images into the MACCO forward pass with unchanged captions and masks. Preserve the clean image tensor as the image reconstruction target before computing the four original losses.


## 03.04 · One-batch checks

Inspect the same image before and after noise. Confirm the encoder input changes while the reconstruction target stays clean. Sigma zero should reproduce baseline loss within numerical tolerance; positive sigma should keep all terms finite.


## 03.05 · Independent training

Reload original OpenAI CLIP weights; initialize fresh predictors and optimizer. Train five full epochs with the baseline row set, common batch size, and original four losses. Log training sigma, seed, loss terms, rows, and updates.


## 03.06 · Evaluate and save

Evaluate both clean and fixed noisy cases with the shared evaluation functions. Save `pixel_noise_comparison.csv` with separate clean and noisy deltas against the four-loss baseline on matching cases.
